In [ ]:
import spyro
from firedrake import VTKFile
from pathlib import Path
import numpy as np
import segyio
import matplotlib.pyplot as plt
from matplotlib.transforms import Affine2D
from firedrake.pyplot import triplot
from IPython import get_ipython

if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

degree = 4
frequency = 1.963
final_time = 1.0

output_dir = Path("results") / "case_large_2d_without_bathymetry"
output_dir.mkdir(parents=True, exist_ok=True)

dictionary = {}
dictionary["options"] = {
    "cell_type": "T",  # simplexes such as triangles or tetrahedra (T) or quadrilaterals (Q)
    "variant": "lumped",  # lumped, equispaced or DG, default is lumped
    "degree": degree,  # p order
    "dimension": 2,  # dimension
}
dictionary["parallelism"] = {
    "type": "automatic",  # options: automatic (same number of cores for evey processor) or spatial
}
dictionary["mesh"] = {
    "length_z": 12.0,  # depth in km - always positive   # Como ver isso sem ler a malha?
    "length_x": 30.0,  # width in km - always positive
    "length_y": 0.0,  # thickness in km - always positive
    "output_filename": str(output_dir / "mesh.msh"),
    "cells_per_wavelength": 3.0,
    "frequency": frequency,
    "segy_velocity_model": "case_large_2d_without_bathymetry.segy",
    "mesh_type": "gmsh_mesh",
    "grade": 0.05,
    "apply_winslow": False,
    "structured_mesh": False,
    "min_element_size": 0.214,  # km: 214 m; target ~8000 structured quads
    "unstructured_quad": False,  # notebook selector; requires native backend support
    "winslow_iterations": 2000,
    "vp_water": 1.5,
    "water_interface": False,
    "water_search_value": 1.5,
}

# Mesh modes (structured_mesh, apply_winslow, unstructured_quad):
# Unstructured triangles/tetrahedra: (False, False, False).
# Structured quadrilaterals/hexahedra: (True, False, False).
# Winslow-adapted structured quads/hexes: (True, True, False).
# Unstructured quads: (False, False, True), 2D only, with a supporting backend.
# PR #359's reviewed public code has no native unstructured-quad switch.
# The check below prevents a silently ignored request; it does not add a mesher.
if dictionary["mesh"]["structured_mesh"] and dictionary["mesh"]["unstructured_quad"]:
    raise ValueError("Choose structured_mesh or unstructured_quad, not both.")
if dictionary["mesh"]["apply_winslow"] and not dictionary["mesh"]["structured_mesh"]:
    raise ValueError("This Winslow implementation requires structured_mesh=True.")
if dictionary["mesh"]["unstructured_quad"]:
    if dictionary["options"]["dimension"] != 2:
        raise NotImplementedError(
            "unstructured_quad selects 2D cells, not unstructured 3D hexahedra. "
            "PR #359 supports structured hexahedra or unstructured tetrahedra."
        )
    import inspect
    from spyro.meshing.meshing_parameters import MeshingParameters
    from spyro.meshing.meshing_functions import AutomaticMesh
    try:
        _parser_source = inspect.getsource(MeshingParameters.__init__)
        _mesher_source = inspect.getsource(AutomaticMesh.create_gmsh_2D_mesh)
    except (OSError, TypeError, AttributeError) as exc:
        raise RuntimeError("Cannot verify native unstructured-quad support.") from exc
    _native_quad_key = next((
        key for key in ("unstructured_quad", "unstructured_quad_mesh")
        if (f'"{key}"' in _parser_source or f"'{key}'" in _parser_source)
        and key in _mesher_source
    ), None)
    if _native_quad_key is None:
        raise NotImplementedError(
            "The installed Spyro Gmsh backend does not expose a recognized "
            "unstructured-quad option. It is absent from the reviewed PR #359. "
            "Keep unstructured_quad=False or use a backend that implements it."
        )
    dictionary["mesh"][_native_quad_key] = True
# Keep the finite-element family consistent with the selected mesh topology.
dictionary["options"]["cell_type"] = (
    "Q" if (dictionary["mesh"]["structured_mesh"]
            or dictionary["mesh"]["unstructured_quad"]) else "T"
)

dictionary["acquisition"] = {
    "source_type": "ricker",
    "source_locations": [(-0.01, 15.0)],
    "frequency": frequency,
    "delay": 1.0 / frequency,
    "delay_type": "time",
    "receiver_locations": spyro.create_transect((-0.1, 1.5), (-0.1, 28.5), 100),
}
dictionary["time_axis"] = {
    "initial_time": 0.0,  # Initial time for event
    "final_time": final_time,  # Final time for event
    "dt": 0.0005,  # timestep size
    "amplitude": 1,  # the Ricker has an amplitude of 1.
    "output_frequency": 100,  # how frequently to output solution to pvds
    "gradient_sampling_frequency": 1,  # how frequently to save solution to RAM
}
dictionary["inversion"] = {
    "perform_fwi": True,  # switch to true to make a FWI
    "initial_guess_model_file": None,
    "shot_record_file": None,
}
dictionary["visualization"] = {
    "forward_output": True,
    "forward_output_filename": str(output_dir / "forward_output.pvd"),
}

def plot_problem():
    # Read only for visualization; the propagator still loads its SEG-Y normally.
    mesh_settings = dictionary["mesh"]
    dimension = dictionary["options"]["dimension"]
    lz, lx, ly = (mesh_settings[key] for key in ("length_z", "length_x", "length_y"))
    sources = np.asarray(dictionary["acquisition"]["source_locations"])
    receivers = np.asarray(dictionary["acquisition"]["receiver_locations"])
    model_path = Path(mesh_settings["segy_velocity_model"])
    if not model_path.is_file():
        raise FileNotFoundError(f"Keep {model_path.name} beside this notebook and open Jupyter in that folder.")
    with segyio.open(str(model_path), "r", ignore_geometry=True) as segy:
        traces = np.asarray(segy.trace.raw[:], dtype=np.float32)
        if dimension == 3:
            inlines = np.asarray(segy.attributes(segyio.TraceField.INLINE_3D)[:])
            crosslines = np.asarray(segy.attributes(segyio.TraceField.CROSSLINE_3D)[:])
            ny, nx = len(np.unique(inlines)), len(np.unique(crosslines))
            if nx * ny != len(traces):
                raise ValueError("The SEG-Y inline/crossline grid is not a complete 3D volume.")
            order = np.lexsort((crosslines, inlines))
            if len(np.unique(np.column_stack((inlines, crosslines)), axis=0)) != len(traces):
                raise ValueError("Duplicate inline/crossline positions in the SEG-Y volume.")
            traces = traces[order].reshape(ny, nx, traces.shape[1])
        else:
            nx, ny = len(traces), 1
    x = np.linspace(0.0, lx, nx)
    z = np.linspace(0.0, -lz, traces.shape[-1])
    water = np.isclose(traces, mesh_settings["water_search_value"], rtol=0.0, atol=1e-6)
    if not np.all(water[..., 0]) or not np.all((~water).any(axis=-1)):
        raise ValueError("Expected a surface water layer followed by non-water in every trace.")
    first_solid = np.argmax(~water, axis=-1)
    # The displayed raster interface lies halfway between water and solid samples.
    interface_z = 0.5 * (z[first_solid - 1] + z[first_solid])
    if dimension == 3:
        y = np.linspace(0.0, ly, ny)
        iy = int(np.argmin(np.abs(y - sources[0, 2])))
        section = traces[iy].T
        section_interface = interface_z[iy]
        section_label = f"; section Y={y[iy]:g} km"
    else:
        section, section_interface = traces.T, interface_z
        section_label = ""
    fig, ax = plt.subplots(figsize=(11, max(4.0, min(8.0, 11 * lz / lx))))
    image = ax.pcolormesh(x, z, section, shading="nearest", rasterized=True)
    fig.colorbar(image, ax=ax, label="Vp (km/s)")
    ax.plot(x, section_interface, linewidth=1.2, label="SEG-Y water interface")
    ax.scatter(receivers[:, 1], receivers[:, 0], marker="v", s=12, label="Receivers", zorder=4)
    ax.scatter(sources[:, 1], sources[:, 0], marker="*", s=150, label="Source", zorder=5)
    ax.set(xlim=(0.0, lx), ylim=(-lz, 0.0), xlabel="X (km)", ylabel="Z (km)",
           title=f"{model_path.stem.replace('_', ' ')}{section_label}")
    ax.set_aspect("equal")
    ax.legend(loc="lower right")
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    if dimension == 3:
        xx, yy = np.meshgrid(x, y, indexing="xy")
        fig = plt.figure(figsize=(10, 7))
        ax = fig.add_subplot(111, projection="3d")
        ax.plot_surface(interface_z, xx, yy, alpha=0.65, rcount=ny, ccount=nx)
        ax.scatter(receivers[:, 0], receivers[:, 1], receivers[:, 2], marker="v", s=12, label="Receivers")
        ax.scatter(sources[:, 0], sources[:, 1], sources[:, 2], marker="*", s=150, label="Source")
        ax.set(xlim=(-lz, 0.0), ylim=(0.0, lx), zlim=(0.0, ly),
               xlabel="Z (km)", ylabel="X (km)", zlabel="Y (km)",
               title="Full SEG-Y water interface: water above, solid below")
        ax.set_box_aspect((lz, lx, ly), zoom=0.8)
        ax.view_init(elev=25, azim=-60, vertical_axis="x")
        ax.legend()
        fig.tight_layout()
        plt.show()
        plt.close(fig)


def plot_generated_mesh(real_wave):
    # Plot the actual mesh used by Spyro, not the SEG-Y sampling grid.
    mesh = real_wave.mesh
    if mesh is None:
        raise RuntimeError("Spyro did not generate a mesh. Check the meshing settings and backend.")
    dimension = dictionary["options"]["dimension"]
    gdim = mesh.geometric_dimension
    gdim = gdim() if callable(gdim) else gdim
    if gdim != dimension:
        raise RuntimeError(f"Requested {dimension}D, but the installed backend generated a {gdim}D mesh.")
    points = np.real(mesh.coordinates.dat.data_ro_with_halos)
    lower, upper = points.min(axis=0), points.max(axis=0)
    sources = np.asarray(dictionary["acquisition"]["source_locations"])
    receivers = np.asarray(dictionary["acquisition"]["receiver_locations"])
    if dimension == 2:
        fig, ax = plt.subplots(figsize=(11, max(4.0, min(8.0, 11 * (upper[0]-lower[0]) / (upper[1]-lower[1])))))
        collections = triplot(mesh, axes=ax, interior_kw={"linewidths": 0.25},
                             boundary_kw={"linewidths": 0.7})
        # Swap only the displayed axes: horizontal X and vertical Z.
        # The Firedrake mesh coordinates and cell connectivity are NOT changed.
        transform = Affine2D().from_values(0, 1, 1, 0, 0, 0) + ax.transData
        for collection in collections:
            collection.set_transform(transform)
            collection.set_rasterized(True)
        ax.scatter(receivers[:, 1], receivers[:, 0], marker="v", s=12, label="Receivers", zorder=4)
        ax.scatter(sources[:, 1], sources[:, 0], marker="*", s=150, label="Source", zorder=5)
        ax.set(xlim=(lower[1], upper[1]), ylim=(lower[0], upper[0]), xlabel="X (km)", ylabel="Z (km)")
        ax.set_aspect("equal")
    else:
        fig = plt.figure(figsize=(10, 7))
        ax = fig.add_subplot(111, projection="3d")
        triplot(mesh, axes=ax, boundary_kw={"linewidths": 0.25, "alpha": 0.25})
        ax.scatter(receivers[:, 0], receivers[:, 1], receivers[:, 2], marker="v", s=12, label="Receivers")
        ax.scatter(sources[:, 0], sources[:, 1], sources[:, 2], marker="*", s=150, label="Source")
        ax.set(xlim=(lower[0], upper[0]), ylim=(lower[1], upper[1]), zlim=(lower[2], upper[2]),
               xlabel="Z (km)", ylabel="X (km)", zlabel="Y (km)")
        ax.set_box_aspect(upper - lower, zoom=0.8)
        ax.view_init(elev=25, azim=-60, vertical_axis="x")
    cell = mesh.ufl_cell().cellname
    cell = cell() if callable(cell) else cell
    suffix = " (exterior facets)" if dimension == 3 else ""
    ax.set_title(f"Generated {cell} mesh: {mesh.num_cells():,} cells{suffix}")
    ax.legend(handles=[ax.collections[-2], ax.collections[-1]], labels=["Receivers", "Source"], loc="best")
    fig.tight_layout()
    plt.show()
    plt.close(fig)


def test_real_shot_record_generation_parallel():
    plot_problem()
    real_wave = spyro.AcousticWave(dictionary=dictionary)
    real_wave.set_initial_velocity_model(
        new_file="case_large_2d_without_bathymetry.segy", fast_interpolate=True
    )
    plot_generated_mesh(real_wave)
    VTKFile(str(output_dir / "vp.pvd")).write(real_wave.c)
    real_wave.forward_solve()
    spyro.io.save_shots(real_wave, file_name=str(output_dir / "shot_record_"))
    return real_wave


if __name__ == "__main__":
    real_wave = test_real_shot_record_generation_parallel()
